# Real-Time Fraud Detection: Handling Class Imbalance Experiments

## 1. Why Accuracy is Misleading in Fraud Detection
In a dataset where 99.83% of transactions are legitimate and 0.17% are fraudulent:
- A trivial model predicting `0` (legitimate) for every transaction achieves **99.83% Accuracy**, but **0% Recall**.
- All fraud slips through undetected, resulting in massive financial losses.

## 2. Imbalance Mitigation Strategies Compared
We compare four approaches on the exact same unseen validation holdout:
1. **Unweighted Baseline**: Standard empirical risk minimization.
2. **Cost-Sensitive Class Weighting**: Penalizing fraud false negatives inversely proportional to class frequencies.
3. **Random Under-Sampling (RUS)**: Down-sampling the majority class on the training set.
4. **SMOTE**: Synthesizing minority fraud examples along nearest-neighbor line segments in feature space.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.data.loader import load_credit_card_data
from src.data.split import time_based_split
from src.features.engineer import TransactionFeatureEngineer
from src.models.resampling import apply_random_undersampling, apply_smote
from src.models.evaluator import evaluate_predictions

# 1. Load data & Leakage-free temporal split
df = load_credit_card_data('../data/raw/creditcard.csv')
train_df, val_df, test_df = time_based_split(df, val_size=0.15, test_size=0.15)

# 2. Feature engineering fitted ONLY on training split
fe = TransactionFeatureEngineer()
fe.fit(train_df.drop(columns=['Class']))

X_train = fe.transform(train_df.drop(columns=['Class']))
y_train = train_df['Class']

X_val = fe.transform(val_df.drop(columns=['Class']))
y_val = val_df['Class']

print(f"Training set: {len(X_train):,} (Frauds: {y_train.sum()})")
print(f"Validation set: {len(X_val):,} (Frauds: {y_val.sum()})")

In [ ]:
# Experiment runner comparing methods
results = []

# Approach 1: Unweighted Baseline
m1 = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
m1.fit(X_train, y_train)
p1 = m1.predict_proba(X_val)[:, 1]
r1 = evaluate_predictions(y_val, p1, threshold=0.5)
results.append({'Strategy': '1. Unweighted Baseline', **r1})

# Approach 2: Cost-Sensitive Class Weighting
m2 = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])
m2.fit(X_train, y_train)
p2 = m2.predict_proba(X_val)[:, 1]
r2 = evaluate_predictions(y_val, p2, threshold=0.5)
results.append({'Strategy': '2. Class Weighting (Balanced)', **r2})

# Approach 3: Random Under-Sampling (on Train only!)
X_train_rus, y_train_rus = apply_random_undersampling(X_train, y_train, sampling_strategy=0.1, random_state=42)
m3 = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
m3.fit(X_train_rus, y_train_rus)
p3 = m3.predict_proba(X_val)[:, 1]
r3 = evaluate_predictions(y_val, p3, threshold=0.5)
results.append({'Strategy': '3. Random Under-Sampling', **r3})

# Approach 4: SMOTE (on Train only!)
X_train_smote, y_train_smote = apply_smote(X_train, y_train, sampling_strategy=0.05, random_state=42)
m4 = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
m4.fit(X_train_smote, y_train_smote)
p4 = m4.predict_proba(X_val)[:, 1]
r4 = evaluate_predictions(y_val, p4, threshold=0.5)
results.append({'Strategy': '4. SMOTE', **r4})

df_results = pd.DataFrame(results)[['Strategy', 'pr_auc', 'recall', 'precision', 'f1', 'total_cost']]
df_results

## 3. Visualizing Performance Trade-offs
Comparing PR-AUC, Recall, Precision, and Total Financial Cost across imbalance handling techniques.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# PR-AUC
sns.barplot(data=df_results, x='Strategy', y='pr_auc', ax=axes[0], palette='Blues_r')
axes[0].set_title('PR-AUC Comparison', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Precision vs Recall
df_melt = pd.melt(df_results, id_vars=['Strategy'], value_vars=['recall', 'precision'], var_name='Metric', value_name='Score')
sns.barplot(data=df_melt, x='Strategy', y='Score', hue='Metric', ax=axes[1], palette=['#d9534f', '#2b5c8f'])
axes[1].set_title('Precision vs Recall Trade-off', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

# Total Business Cost
sns.barplot(data=df_results, x='Strategy', y='total_cost', ax=axes[2], palette='Reds_r')
axes[2].set_title('Total Business Loss ($)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Key Takeaways on Imbalance Handling
1. **Never Assume SMOTE is Universally Superior**: In high-dimensional latent spaces, SMOTE can generate synthetic points in low-density ambiguous border regions, inflating false positive rates.
2. **Class Weighting & Algorithmic Loss Scaling**: Cost-sensitive weighting directly optimizes gradient penalization without corrupting empirical feature covariances.
3. **Threshold Tuning is More Effective than Extreme Resampling**: As shown in step 5, probability threshold optimization achieves the desired precision-recall balance far more cleanly than aggressive data distortion.